In [ ]:
# Install gwexpy with pinned versions of core dependencies for reproducibility on Colab

%pip install -q "gwexpy[all]" "gwpy<5.0.0" "numpy<2.0.0" "scipy<1.13.0" "astropy<7.0.0"

# SegmentTable: Basics

:::{tip}
For a more comprehensive guide including visualization, GravitySpy integration, and advanced operations, see the [Table / Segment User Guide](intro_table.ipynb).
:::

## GWpy base classes and gwexpy extensions

Segment analysis builds on GWpy types. Each row span is still represented by `gwpy.segments.Segment`, and payload columns can hold GWpy `TimeSeries` or `FrequencySeries` objects directly.

On top of that, gwexpy adds `SegmentTable` as an extension of `gwpy.table.Table`, plus lazy payload handling via `SegmentCell`, and table-oriented batch helpers such as `apply()`, `map()`, `crop()`, and `asd()`. In practice, you keep GWpy base classes for the actual data objects while gwexpy adds the workflow layer for processing many segments together.


# SegmentTable: 基本

:::{tip}
可視化、GravitySpyとの連携、高度な操作を含むより包括的なガイドについては、[テーブル / セグメント ユーザーガイド](intro_table.ipynb)を参照してください。
:::

# SegmentTable の基本

このチュートリアルでは、**GWexpy** におけるセグメント（時間区間）ベースの解析コンテナである `SegmentTable` を紹介します。通常の `EventTable` とは異なり、`SegmentTable` は区間のメタデータだけでなく、対応するペイロード（TimeSeries や ASD など）を遅延ロード（Lazy loading）形式で保持することに特化しています。

## GWpy 基盤クラスと gwexpy 拡張

セグメント解析の土台は GWpy にあります。各行の区間は `gwpy.segments.Segment` をそのまま使い、ペイロード列には GWpy の `TimeSeries` や `FrequencySeries` をそのまま載せられます。

そのうえで gwexpy は、`gwpy.table.Table` を拡張した `SegmentTable` と、遅延ロード可能な `SegmentCell` を提供し、`apply()` / `map()` / `crop()` / `asd()` のような表ベースの一括処理を追加しています。つまり、GWpy の基本型を保ったまま、複数セグメントをまとめて処理するワークフローを gwexpy が補強する構成です。


In [ ]:
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

import warnings

with warnings.catch_warnings():
    warnings.simplefilter('ignore')

    import numpy as np
    from gwpy.segments import Segment

    from gwexpy.table import SegmentTable

    # 1. Create simple segments
    segs = [Segment(0, 4), Segment(4, 8), Segment(8, 12)]
    st = SegmentTable.from_segments(segs, label=["A", "B", "C"])
    st


## Delayed Loading with SegmentCell

You can add payload columns that are only loaded when needed. This is useful for handling large datasets.

## SegmentCell による遅延ロード

必要になるまでデータをロードしない「ペイロード列」を追加できます。これは巨大なデータのバッチ処理に非常に有効です。

In [ ]:
def my_loader():
    # Simulate loading data
    print("Loading series...")
    from gwpy.timeseries import TimeSeries
    return TimeSeries(np.random.randn(128), sample_rate=32)

# Add a payload column with a loader (sequence of callables)
st.add_series_column("raw", loader=[my_loader]*len(st), kind="timeseries")
st

## Row-wise Processing

`SegmentTable` provides an `apply()` method to process each row and collect results into new columns.

## 行単位の処理

`SegmentTable` は `apply()` メソッドを提供し、各行を処理して新しい列として統合できます。

In [ ]:
def process_row(row):
    span = row["span"]
    return {"duration": float(span[1] - span[0]), "valid": True}

st2 = st.apply(process_row)
st2.display()

## Fetch and Conversion

You can explicitly load lazy cells with `fetch()` or `materialize()`. Converting to pandas gives you a standard DataFrame for meta columns.

## 明示的ロードとデータ変換

`fetch()` や `materialize()` を使ってデータを明示的にロードできます。`to_pandas()` を使うと、通常の pandas DataFrame として扱えます。

In [ ]:
st2.fetch()
df = st2.to_pandas()
df.head()